# Free beamtime on LoKI: starter notebook

No exercise here: **your own question**. Write it down first (below), as in a beamtime proposal at ESS: what do you
want to learn, on which sample, and why neutrons and LoKI can answer it. Then plan your measurements (which
references does your analysis need? how long? in which configuration?), measure in NICOS (button *Open NICOS*), and
analyze your files here. Your own samples can be designed with the *Sample builder*: they appear on the sample changer
of NICOS.

**Your proposal**: choose a proposal number (e.g. your group number, 1000-9999) and enter it in the *Experiment* panel
of NICOS; write it below in `PROPOSAL`. Your files are then in `data/loki/<year>/<proposal>/raw`.

**Tools**
- `rows`: your logbook (one line per count: file, kind, title, settings);
- `red.reduce(sample, background=..., empty_beam=..., banks=[...])`: $I(Q)$ per detector bank, with the official ESS
  software (esssans); `red.combine(result, thickness_cm)`: banks merged into rear/middle/front, in cm⁻¹;
- `la.guinier(q, I, dI)`, `la.sphere_fit(q, I, dI)`: Guinier analysis and the sphere model of sasmodels (SasView);
- `h5py` to look inside a NeXus file; `scipy.optimize` for your own fits.

## My question

(what, on which sample, why; what result would answer it)

## My plan

(measurements, references, configuration, counting times, and why)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))
import nb_tools

## 1. Your sample

The sample changer of LoKI holds the samples of the library (in NICOS, `sample_info()` describes the one mounted). For your own question, design a sample with the sample builder: particles of the kinds you choose (spheres, cylinders, core-shell spheres, as many kinds as you want) in a H2O/D2O solvent. It is saved with your proposal and appears on the sample changer of NICOS: `move(sample_holder, 'its_name')`.

In [ ]:
nb_tools.sample_builder("loki", "1000")

## 2. Your measurements, in NICOS

1. Tab *Experiment*: give your *Proposal ID* (the number of PROPOSAL below), then **Apply** (first: your own samples are kept with your proposal, NICOS finds them only then).
2. **Mount your sample** (tab *Instrument interaction*, command line): `move(sample_holder, 'its_name')`, the short name of a sample of the library or of the one you made in part 1. `sample_info()` describes the sample mounted; for a name NICOS does not know, it lists the samples available. Every count measures the sample mounted at that moment.
3. **Record**: `with nexusfile_open('my title'): count(t=60)`; your files go to the folder of your proposal (below). The measurements and their counting times are your decisions: no budget.

In [ ]:
nb_tools.nicos("loki")

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))     # the modules of the LoKI twin (folder loki/)

In [ ]:
from loki_paths import TWIN_RUN_ROOT
import datetime
PROPOSAL = "1000"          # your proposal number (the one entered in NICOS)
FOLDER = f"{TWIN_RUN_ROOT}/data/loki/{datetime.date.today().year}/{PROPOSAL}/raw"

In [ ]:
import sys, os, json, glob, datetime, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath("../.."))
import numpy as np
import scipp as sc
import h5py
import matplotlib.pyplot as plt
from loki_paths import TWIN_RUN_ROOT
import loki_reduction as red
import loki_analysis as la
import loki_samples as ls
import nb_tools
nb_tools.need_measurements(FOLDER, "loki")   # stops here while there is no file

In [ ]:
rows = []
with open(f"{FOLDER}/log.jsonl", encoding="utf-8") as f:
    for line in f:
        entry = json.loads(line)
        d = entry["details"]
        if entry["action"] != "count" or not d.get("file"):
            continue
        s = d["state"]
        rows.append(dict(file=os.path.basename(d["file"]), kind=red.kind(s), title=d["title"], t_s=d["t"],
                         collimation=s["collimation"], pulse_skipping=s.get("pulse_skipping", "off"),
                         detector_m=s["detector_distance"], sample=s["sample"]))
def show_table(rows):                                  # a list of dicts, printed as a table
    if not rows:
        print("no measurement in the logbook"); return
    cols = list(rows[0])
    width = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in cols}
    print("  ".join(c.ljust(width[c]) for c in cols))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(width[c]) for c in cols))

show_table(rows)

## Your analysis

## My answer

(the answer to your question, with uncertainties; what you would do next)

## Submit your work

When you are done: save this notebook (*Save*), then submit. The file for your teacher is made from your measurements and your notebook.

In [ ]:
import submit
_ = submit.Submit("loki/exercises/free_beamtime", "free", title="Free beamtime on LoKI")